# QAOA-GPT inference demo

In [1]:
from model_interface import QAOA_GPT

In [2]:
import pandas as pd
import networkx as nx
import random
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


## Loading a model

In [3]:
qaoa_gpt_n10_obj = QAOA_GPT(
    model_ckpt='nanoGPT/out-10nodes_netlsd/gpt_ckpt_2000_netlsd_ar_0_95652__er_0_0.pt',
    data_dir='nanoGPT/data/10nodes_netlsd', # to take meta.pkl file 
    config_file='nanoGPT/data/10nodes_netlsd/train_adapt_gpt_config.py',
    temp_folder='temp_data',
    device='cuda'
)

Loading config from: nanoGPT/data/10nodes_netlsd/train_adapt_gpt_config.py
Initiating nanoGPT model with padding support
number of parameters: 11.68M


In [4]:
# qaoa_gpt_n10_obj = QAOA_GPT(
#     model_ckpt='nanoGPT/models/n10w_qaoa_mixer/ckpt_16000_gemb__ar_0_96584__er_0_0.pt',
#     data_dir='nanoGPT/models/n10w_qaoa_mixer/data', # to take meta.pkl file 
#     config_file='nanoGPT/models/n10w_qaoa_mixer/data/train_adapt_gpt_config.py',
#     temp_folder='temp_data',
#     device='cuda'
# )

## Generating random graphs

In [5]:
def add_weights_to_nx_graph(nx_graph):
    for u, v in nx_graph.edges():
        cur_weight = round(random.uniform(0, 1), 2)
        while cur_weight == 0:
            cur_weight = round(random.uniform(0, 1), 2)
        nx_graph[u][v]['weight'] = cur_weight
    return nx_graph

In [6]:
tqdm.pandas()

Modify this nodes here to match the model before run

In [7]:
n_graphs = 5
n_nodes = 10

In [8]:
graphs_edgelist_list_dict = dict()

er_graphs_edgelist_list_dict = dict()
for i in range(n_graphs):
    p = random.randrange(6,9) / 10
    cur_graph = nx.erdos_renyi_graph(
        n=n_nodes,
        p=p
    )
    
    er_graphs_edgelist_list_dict[f'er_graph_{i}'] = add_weights_to_nx_graph(cur_graph)

graphs_edgelist_list_dict.update(er_graphs_edgelist_list_dict)

In [9]:
graphs_edgelist_list_dict['er_graph_2'].edges(data=True)

EdgeDataView([(0, 1, {'weight': 0.68}), (0, 3, {'weight': 0.93}), (1, 3, {'weight': 0.75}), (1, 5, {'weight': 0.68}), (1, 7, {'weight': 0.57}), (1, 9, {'weight': 0.81}), (2, 3, {'weight': 0.57}), (2, 5, {'weight': 0.53}), (2, 6, {'weight': 0.83}), (2, 7, {'weight': 0.89}), (2, 9, {'weight': 0.19}), (3, 4, {'weight': 0.06}), (3, 7, {'weight': 0.82}), (4, 5, {'weight': 0.64}), (4, 6, {'weight': 0.96}), (4, 7, {'weight': 0.89}), (4, 8, {'weight': 0.24}), (5, 7, {'weight': 1.0}), (6, 7, {'weight': 0.65}), (6, 8, {'weight': 0.44}), (7, 9, {'weight': 0.65}), (8, 9, {'weight': 0.1})])

## Generate circuits with QAOA-GPT model

In [10]:
embedding_method = qaoa_gpt_n10_obj.embedding_method
print(f"Using embedding method: {embedding_method}")

qaoa_gpt_circ_df = qaoa_gpt_n10_obj.generate_circ_from_nx(
    graphs_edgelist_list_dict,
    # calculate_classic_maxcut=True, # to create col enery_gurobi. Default:True
    n_samples_per_batch=50, # max number of distinct graphs in a batch
    num_samples=5, # number of samples to draw
    max_new_tokens=150, # number of tokens generated in each sample
    temperature=0.1, # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
    top_k=200, # retain only the top_k most likely tokens, clamp others to have 0 probability
    embedding_method=embedding_method,
)

Using embedding method: netlsd


Preparing graphs...:   0%|          | 0/5 [00:00<?, ?it/s]

Restricted license - for non-production use only - expires 2027-11-29


Preparing graphs...: 100%|██████████| 5/5 [00:00<00:00, 57.50it/s]


Performing FEATHER embedding


NetLSD: 100%|██████████| 5/5 [00:00<00:00, 379.55it/s]
Inference. Current batch: n_edges: 33, n_graphs: 1: 100%|██████████| 5/5 [00:02<00:00,  1.77it/s]


In [11]:
qaoa_gpt_n10_obj.graphs_nx_df[:1]

,graph_id,elist,n_nodes,energy_gurobi,token_seq_round_d2,edgelist_list_len,approx_ratio,label,edgelist_json,has_emb
0,er_graph_0,"[(1, 2, 0.94), (1, 3, 0.75), (1, 5, 0.38), (1, 7, 0.31), (1, 8, 0.44), (1, 9, 0.38), (2, 3, 0.1), (2, 4, 0.25), (2, 5, 0.99), (2, 8, 0.29), (2, 10, 0.39), (3, 4, 0.6), (3, 6, 0.05), (3, 7, 0.03), (3, 8, 0.38), (3, 9, 0.85), (3, 10, 0.23), (4, 7, 0.35), (4, 8, 0.95), (5, 6, 0.44), (5, 8, 0.41), (5, 9, 0.31), (5, 10, 0.73), (6, 8, 0.21), (6, 10, 0.39), (7, 8, 0.53), (7, 9, 0.86), (7, 10, 0.93), (8, 10, 0.06), (9, 10, 0.14)]",10,-10.14,"[bos, (1, 2), 0.94, (1, 3), 0.75, (1, 5), 0.38, (1, 7), 0.31, (1, 8), 0.44, (1, 9), 0.38, (2, 3), 0.1, (2, 4), 0.25, (2, 5), 0.99, (2, 8), 0.29, (2, 10), 0.39, (3, 4), 0.6, (3, 6), 0.05, (3, 7), 0.03, (3, 8), 0.38, (3, 9), 0.85, (3, 10), 0.23, (4, 7), 0.35, (4, 8), 0.95, (5, 6), 0.44, (5, 8), 0.41, (5, 9), 0.31, (5, 10), 0.73, (6, 8), 0.21, (6, 10), 0.39, (7, 8), 0.53, (7, 9), 0.86, (7, 10), 0.93, (8, 10), 0.06, (9, 10), 0.14, end_of_graph]",30,None,test_interactive,"[[1, 2, 0.94], [1, 3, 0.75], [1, 5, 0.38], [1, 7, 0.31], [1, 8, 0.44], [1, 9, 0.38], [2, 3, 0.1], [2, 4, 0.25], [2, 5, 0.99], [2, 8, 0.29], [2, 10, 0.39], [3, 4, 0.6], [3, 6, 0.05], [3, 7, 0.03], [3, 8, 0.38], [3, 9, 0.85], [3, 10, 0.23], [4, 7, 0.35], [4, 8, 0.95], [5, 6, 0.44], [5, 8, 0.41], [5, 9, 0.31], [5, 10, 0.73], [6, 8, 0.21], [6, 10, 0.39], [7, 8, 0.53], [7, 9, 0.86], [7, 10, 0.93], [8, 10, 0.06], [9, 10, 0.14]]",True


In [12]:
qaoa_gpt_n10_obj.feather_par_emb[0][:10]

array([0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99, 0.99])

In [13]:
# qaoa_gpt_n10_obj.meta

In [14]:
# qaoa_gpt_n10_obj.emb_graph_id_to_idx_dict

In [15]:
sample_gr = graphs_edgelist_list_dict['er_graph_0'].edges(data=True)
sample_gr

EdgeDataView([(0, 1, {'weight': 0.94}), (0, 2, {'weight': 0.75}), (0, 4, {'weight': 0.38}), (0, 6, {'weight': 0.31}), (0, 7, {'weight': 0.44}), (0, 8, {'weight': 0.38}), (1, 2, {'weight': 0.1}), (1, 3, {'weight': 0.25}), (1, 4, {'weight': 0.99}), (1, 7, {'weight': 0.29}), (1, 9, {'weight': 0.39}), (2, 3, {'weight': 0.6}), (2, 5, {'weight': 0.05}), (2, 6, {'weight': 0.03}), (2, 7, {'weight': 0.38}), (2, 8, {'weight': 0.85}), (2, 9, {'weight': 0.23}), (3, 6, {'weight': 0.35}), (3, 7, {'weight': 0.95}), (4, 5, {'weight': 0.44}), (4, 7, {'weight': 0.41}), (4, 8, {'weight': 0.31}), (4, 9, {'weight': 0.73}), (5, 7, {'weight': 0.21}), (5, 9, {'weight': 0.39}), (6, 7, {'weight': 0.53}), (6, 8, {'weight': 0.86}), (6, 9, {'weight': 0.93}), (7, 9, {'weight': 0.06}), (8, 9, {'weight': 0.14})])

In [16]:
len(graphs_edgelist_list_dict['er_graph_0'].edges(data=True))

30

The graph after prediction is shifted by 1 unit. For example, in NetworkX the edge is (0, 2, 0.36), but in this DataFrame it becomes (1, 3, 0.36)

In [17]:
qaoa_gpt_circ_df[:1]

,graph,n_edges,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_gurobi,label,graph_w_jl,graph_weight_norm
0,"[(1, 2), 0.94, (1, 3), 0.75, (1, 5), 0.38, (1, 7), 0.31, (1, 8), 0.44, (1, 9), 0.38, (2, 3), 0.1, (2, 4), 0.25, (2, 5), 0.99, (2, 8), 0.29, (2, 10), 0.39, (3, 4), 0.6, (3, 6), 0.05, (3, 7), 0.03, (3, 8), 0.38, (3, 9), 0.85, (3, 10), 0.23, (4, 7), 0.35, (4, 8), 0.95, (5, 6), 0.44, (5, 8), 0.41, (5, 9), 0.31, (5, 10), 0.73, (6, 8), 0.21, (6, 10), 0.39, (7, 8), 0.53, (7, 9), 0.86, (7, 10), 0.93, (8, 10), 0.06, (9, 10), 0.14]",30,"[[new_layer_p, 11.0, -2.09, 0.32, new_layer_p, 11.0, -0.39, 0.74, new_layer_p, 11.0, -0.32, 0.74, new_layer_p, 11.0, -0.29, 0.77, new_layer_p, 11.0, -0.25, 0.95, new_layer_p, 11.0, -0.22, 0.94, new_layer_p, 11.0, -0.19, 1.01, new_layer_p, 11.0, -0.11, 1.21, new_layer_p, 11.0, -0.08, 1.3], [new_layer_p, 11.0, -2.09, 0.31, new_layer_p, 11.0, -0.38, 0.77, new_layer_p, 11.0, -0.32, 0.77, new_layer_p, 11.0, -0.29, 0.83, new_layer_p, 11.0, -0.25, 0.96, new_layer_p, 11.0, -0.22, 0.93, new_layer_p, 11.0, -0.19, 1.01, new_layer_p, 11.0, -0.17, 1.21, new_layer_p, 11.0, -0.11, 1.21, new_layer_p, 11.0, -0.08, 1.25], [new_layer_p, 11.0, -2.09, 0.32, new_layer_p, 11.0, -0.39, 0.71, new_layer_p, 11.0, -0.33, 0.71, new_layer_p, 11.0, -0.29, 0.88, new_layer_p, 11.0, -0.25, 0.85, new_layer_p, 11.0, -0.22, 0.85, new_layer_p, 11.0, -0.19, 1.01, new_layer_p, 11.0, -0.12, 1.21, new_layer_p, 11.0, -0.08, 1.3], [new_layer_p, 11.0, -2.09, 0.31, new_layer_p, 11.0, -0.39, 0.71, new_layer_p, 11.0, -0.32, 0.71, new_layer_p, 11.0, -0.29, 0.88, new_layer_p, 11.0, -0.27, 0.85, new_layer_p, 11.0, -0.25, 0.93, new_layer_p, 11.0, -0.22, 0.98, new_layer_p, 11.0, -0.19, 1.08, new_layer_p, 11.0, -0.1, 1.15], [new_layer_p, 11.0, -2.09, 0.31, new_layer_p, 11.0, -0.39, 0.77, new_layer_p, 11.0, -0.32, 0.78, new_layer_p, 11.0, -0.29, 0.85, new_layer_p, 11.0, -0.27, 0.94, new_layer_p, 11.0, -0.24, 0.92, new_layer_p, 11.0, -0.2, 1.01, new_layer_p, 11.0, -0.12, 1.18, new_layer_p, 11.0, -0.08, 1.16]]",[],None,er_graph_0,-10.14,test_interactive,"[[1, 2, 0.94], [1, 3, 0.75], [1, 5, 0.38], [1, 7, 0.31], [1, 8, 0.44], [1, 9, 0.38], [2, 3, 0.1], [2, 4, 0.25], [2, 5, 0.99], [2, 8, 0.29], [2, 10, 0.39], [3, 4, 0.6], [3, 6, 0.05], [3, 7, 0.03], [3, 8, 0.38], [3, 9, 0.85], [3, 10, 0.23], [4, 7, 0.35], [4, 8, 0.95], [5, 6, 0.44], [5, 8, 0.41], [5, 9, 0.31], [5, 10, 0.73], [6, 8, 0.21], [6, 10, 0.39], [7, 8, 0.53], [7, 9, 0.86], [7, 10, 0.93], [8, 10, 0.06], [9, 10, 0.14]]",1.0


In [18]:
qaoa_gpt_circ_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   graph              5 non-null      object 
 1   n_edges            5 non-null      int64  
 2   q_circuits         5 non-null      object 
 3   adapt_circuit      5 non-null      object 
 4   adapt_full_ar      0 non-null      object 
 5   graph_prefix       5 non-null      object 
 6   energy_gurobi      5 non-null      float64
 7   label              5 non-null      object 
 8   graph_w_jl         5 non-null      object 
 9   graph_weight_norm  5 non-null      float64
dtypes: float64(2), int64(1), object(7)
memory usage: 528.0+ bytes


## Evaluate circuits

In [19]:
qaoa_gpt_circ_eval_df = qaoa_gpt_n10_obj.eval_circ_df_jl(qaoa_gpt_circ_df)


===== DEBUG INFO =====
CWD: /home/mrzaizai2k/code_bao/ADAPT_GPT
Command:
/opt/julia-1.12.1/bin/julia -t 4 --project=/home/mrzaizai2k/code_bao/ADAPT_GPT/ADAPT.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/adapt_gpt_eval_energy.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-03-11__23_05_50_df.json /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-03-11__23_05_50_df_jl.json 10 qaoa_double_pool




Julia return code: 0


In [20]:
sample_gr

EdgeDataView([(0, 1, {'weight': 0.94}), (0, 2, {'weight': 0.75}), (0, 4, {'weight': 0.38}), (0, 6, {'weight': 0.31}), (0, 7, {'weight': 0.44}), (0, 8, {'weight': 0.38}), (1, 2, {'weight': 0.1}), (1, 3, {'weight': 0.25}), (1, 4, {'weight': 0.99}), (1, 7, {'weight': 0.29}), (1, 9, {'weight': 0.39}), (2, 3, {'weight': 0.6}), (2, 5, {'weight': 0.05}), (2, 6, {'weight': 0.03}), (2, 7, {'weight': 0.38}), (2, 8, {'weight': 0.85}), (2, 9, {'weight': 0.23}), (3, 6, {'weight': 0.35}), (3, 7, {'weight': 0.95}), (4, 5, {'weight': 0.44}), (4, 7, {'weight': 0.41}), (4, 8, {'weight': 0.31}), (4, 9, {'weight': 0.73}), (5, 7, {'weight': 0.21}), (5, 9, {'weight': 0.39}), (6, 7, {'weight': 0.53}), (6, 8, {'weight': 0.86}), (6, 9, {'weight': 0.93}), (7, 9, {'weight': 0.06}), (8, 9, {'weight': 0.14})])

The eval df add 1 more col is adapt_gpt_energies

Each graph is generate into 5 other graphs, that why we see list of 5 q_circuits and adapt_gpt_energies



In [21]:
qaoa_gpt_circ_eval_df[:1]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 2], 0.9400000000000001, [1, 3], 0.75, [1, 5], 0.38, [1, 7], 0.31, [1, 8], 0.44, [1, 9], 0.38, [2, 3], 0.1, [2, 4], 0.25, [2, 5], 0.99, [2, 8], 0.29, [2, 10], 0.39, [3, 4], 0.6000000000000001, [3, 6], 0.05, [3, 7], 0.03, [3, 8], 0.38, [3, 9], 0.85, [3, 10], 0.23, [4, 7], 0.35000000000000003, [4, 8], 0.9500000000000001, [5, 6], 0.44, [5, 8], 0.41000000000000003, [5, 9], 0.31, [5, 10], 0.73, [6, 8], 0.21, [6, 10], 0.39, [7, 8], 0.53, [7, 9], 0.86, [7, 10], 0.93, [8, 10], 0.06, [9, 10], 0.14]",30,"[[new_layer_p, 11, -2.09, 0.32, new_layer_p, 11, -0.39, 0.74, new_layer_p, 11, -0.32, 0.74, new_layer_p, 11, -0.29, 0.77, new_layer_p, 11, -0.25, 0.9500000000000001, new_layer_p, 11, -0.22, 0.9400000000000001, new_layer_p, 11, -0.19, 1.01, new_layer_p, 11, -0.11, 1.21, new_layer_p, 11, -0.08, 1.3], [new_layer_p, 11, -2.09, 0.31, new_layer_p, 11, -0.38, 0.77, new_layer_p, 11, -0.32, 0.77, new_layer_p, 11, -0.29, 0.8300000000000001, new_layer_p, 11, -0.25, 0.96, new_layer_p, 11, -0.22, 0.93, new_layer_p, 11, -0.19, 1.01, new_layer_p, 11, -0.17, 1.21, new_layer_p, 11, -0.11, 1.21, new_layer_p, 11, -0.08, 1.25], [new_layer_p, 11, -2.09, 0.32, new_layer_p, 11, -0.39, 0.71, new_layer_p, 11, -0.33, 0.71, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.25, 0.85, new_layer_p, 11, -0.22, 0.85, new_layer_p, 11, -0.19, 1.01, new_layer_p, 11, -0.12, 1.21, new_layer_p, 11, -0.08, 1.3], [new_layer_p, 11, -2.09, 0.31, new_layer_p, 11, -0.39, 0.71, new_layer_p, 11, -0.32, 0.71, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.27, 0.85, new_layer_p, 11, -0.25, 0.93, new_layer_p, 11, -0.22, 0.98, new_layer_p, 11, -0.19, 1.08, new_layer_p, 11, -0.1, 1.15], [new_layer_p, 11, -2.09, 0.31, new_layer_p, 11, -0.39, 0.77, new_layer_p, 11, -0.32, 0.78, new_layer_p, 11, -0.29, 0.85, new_layer_p, 11, -0.27, 0.9400000000000001, new_layer_p, 11, -0.24, 0.92, new_layer_p, 11, -0.2, 1.01, new_layer_p, 11, -0.12, 1.18, new_layer_p, 11, -0.08, 1.16]]","[-9.812843512461942, -9.836201397897458, -9.837094902510787, -9.799520672199215, -9.783208107092486]",-10.14


In [22]:
qaoa_gpt_circ_eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   graph_prefix        5 non-null      object 
 1   graph               5 non-null      object 
 2   n_edges             5 non-null      int64  
 3   q_circuits          5 non-null      object 
 4   adapt_gpt_energies  5 non-null      object 
 5   energy_gurobi       5 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 368.0+ bytes


In [23]:
# 3 extract first rows into 5 rows for 5 q_circuits and adapt_gpt_energies
qaoa_gpt_circ_eval_expl_df = qaoa_gpt_circ_eval_df.explode(['adapt_gpt_energies', 'q_circuits']) 

In [24]:
qaoa_gpt_circ_eval_expl_df[:5]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 2], 0.9400000000000001, [1, 3], 0.75, [1, 5], 0.38, [1, 7], 0.31, [1, 8], 0.44, [1, 9], 0.38, [2, 3], 0.1, [2, 4], 0.25, [2, 5], 0.99, [2, 8], 0.29, [2, 10], 0.39, [3, 4], 0.6000000000000001, [3, 6], 0.05, [3, 7], 0.03, [3, 8], 0.38, [3, 9], 0.85, [3, 10], 0.23, [4, 7], 0.35000000000000003, [4, 8], 0.9500000000000001, [5, 6], 0.44, [5, 8], 0.41000000000000003, [5, 9], 0.31, [5, 10], 0.73, [6, 8], 0.21, [6, 10], 0.39, [7, 8], 0.53, [7, 9], 0.86, [7, 10], 0.93, [8, 10], 0.06, [9, 10], 0.14]",30,"[new_layer_p, 11, -2.09, 0.32, new_layer_p, 11, -0.39, 0.74, new_layer_p, 11, -0.32, 0.74, new_layer_p, 11, -0.29, 0.77, new_layer_p, 11, -0.25, 0.9500000000000001, new_layer_p, 11, -0.22, 0.9400000000000001, new_layer_p, 11, -0.19, 1.01, new_layer_p, 11, -0.11, 1.21, new_layer_p, 11, -0.08, 1.3]",-9.812844,-10.14
0,er_graph_0,"[[1, 2], 0.9400000000000001, [1, 3], 0.75, [1, 5], 0.38, [1, 7], 0.31, [1, 8], 0.44, [1, 9], 0.38, [2, 3], 0.1, [2, 4], 0.25, [2, 5], 0.99, [2, 8], 0.29, [2, 10], 0.39, [3, 4], 0.6000000000000001, [3, 6], 0.05, [3, 7], 0.03, [3, 8], 0.38, [3, 9], 0.85, [3, 10], 0.23, [4, 7], 0.35000000000000003, [4, 8], 0.9500000000000001, [5, 6], 0.44, [5, 8], 0.41000000000000003, [5, 9], 0.31, [5, 10], 0.73, [6, 8], 0.21, [6, 10], 0.39, [7, 8], 0.53, [7, 9], 0.86, [7, 10], 0.93, [8, 10], 0.06, [9, 10], 0.14]",30,"[new_layer_p, 11, -2.09, 0.31, new_layer_p, 11, -0.38, 0.77, new_layer_p, 11, -0.32, 0.77, new_layer_p, 11, -0.29, 0.8300000000000001, new_layer_p, 11, -0.25, 0.96, new_layer_p, 11, -0.22, 0.93, new_layer_p, 11, -0.19, 1.01, new_layer_p, 11, -0.17, 1.21, new_layer_p, 11, -0.11, 1.21, new_layer_p, 11, -0.08, 1.25]",-9.836201,-10.14
0,er_graph_0,"[[1, 2], 0.9400000000000001, [1, 3], 0.75, [1, 5], 0.38, [1, 7], 0.31, [1, 8], 0.44, [1, 9], 0.38, [2, 3], 0.1, [2, 4], 0.25, [2, 5], 0.99, [2, 8], 0.29, [2, 10], 0.39, [3, 4], 0.6000000000000001, [3, 6], 0.05, [3, 7], 0.03, [3, 8], 0.38, [3, 9], 0.85, [3, 10], 0.23, [4, 7], 0.35000000000000003, [4, 8], 0.9500000000000001, [5, 6], 0.44, [5, 8], 0.41000000000000003, [5, 9], 0.31, [5, 10], 0.73, [6, 8], 0.21, [6, 10], 0.39, [7, 8], 0.53, [7, 9], 0.86, [7, 10], 0.93, [8, 10], 0.06, [9, 10], 0.14]",30,"[new_layer_p, 11, -2.09, 0.32, new_layer_p, 11, -0.39, 0.71, new_layer_p, 11, -0.33, 0.71, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.25, 0.85, new_layer_p, 11, -0.22, 0.85, new_layer_p, 11, -0.19, 1.01, new_layer_p, 11, -0.12, 1.21, new_layer_p, 11, -0.08, 1.3]",-9.837095,-10.14
0,er_graph_0,"[[1, 2], 0.9400000000000001, [1, 3], 0.75, [1, 5], 0.38, [1, 7], 0.31, [1, 8], 0.44, [1, 9], 0.38, [2, 3], 0.1, [2, 4], 0.25, [2, 5], 0.99, [2, 8], 0.29, [2, 10], 0.39, [3, 4], 0.6000000000000001, [3, 6], 0.05, [3, 7], 0.03, [3, 8], 0.38, [3, 9], 0.85, [3, 10], 0.23, [4, 7], 0.35000000000000003, [4, 8], 0.9500000000000001, [5, 6], 0.44, [5, 8], 0.41000000000000003, [5, 9], 0.31, [5, 10], 0.73, [6, 8], 0.21, [6, 10], 0.39, [7, 8], 0.53, [7, 9], 0.86, [7, 10], 0.93, [8, 10], 0.06, [9, 10], 0.14]",30,"[new_layer_p, 11, -2.09, 0.31, new_layer_p, 11, -0.39, 0.71, new_layer_p, 11, -0.32, 0.71, new_layer_p, 11, -0.29, 0.88, new_layer_p, 11, -0.27, 0.85, new_layer_p, 11, -0.25, 0.93, new_layer_p, 11, -0.22, 0.98, new_layer_p, 11, -0.19, 1.08, new_layer_p, 11, -0.1, 1.15]",-9.799521,-10.14
0,er_graph_0,"[[1, 2], 0.9400000000000001, [1, 3], 0.75, [1, 5], 0.38, [1, 7], 0.31, [1, 8], 0.44, [1, 9], 0.38, [2, 3], 0.1, [2, 4], 0.25, [2, 5], 0.99, [2, 8], 0.29, [2, 10], 0.39, [3, 4], 0.6000000000000001, [3, 6], 0.05, [3, 7], 0.03, [3, 8], 0.38, [3, 9], 0.85, [3, 10], 0.23, [4, 7], 0.35000000000000003, [4, 8], 0.9500000000000001, [5, 6], 0.44, [5, 8], 0.41000000000000003, [5, 9], 0.31, [5, 10], 0.73, [6, 8], 0.21, [6, 10], 0.39, [7, 8], 0.53, [7, 9], 0.86, [7, 10], 0.93, [8, 10], 0.06, [9, 10], 0.14]",30,"[new_layer_p, 11, -2.09, 0.31, new_layer_p, 11, -0.39, 0.77, new_layer_p, 11, -0.32, 0.78, new_layer_p, 11, -0.29

This computes how close each QAOA-GPT circuit’s energy is to the optimal solution (AR — approximation ratio).

If the ratio = 1 → perfect solution.

If <1 → circuit energy is above the optimal energy (since MaxCut is a minimization problem in some conventions, sometimes they flip it; conceptually, ratio shows performance).

In [25]:
(qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] / qaoa_gpt_circ_eval_expl_df['energy_gurobi']).mean()

np.float64(0.9531556197793043)

In [26]:
# on avg, how many layers are there in the predicted circuits
qaoa_gpt_circ_eval_expl_df['q_circuits'].apply(lambda x: x.count('new_layer_p')).mean()

np.float64(9.0)